In [ ]:
# --- Bootstrap cell : à exécuter en premier sur une VM Colab fraîche ---
# Le noyau Colab tourne sur une machine distante avec son propre système de fichiers,
# séparé de ce notebook ouvert dans VS Code. Il ne voit ni ce fichier ni le dépôt qui
# le contient tant qu'on ne les récupère pas explicitement. Le dépôt étant privé, on
# télécharge uniquement les CSV dont ce notebook a besoin (pas tout le dépôt) via
# l'API GitHub authentifiée par token — pas besoin de "Mount Google Drive", ça n'a
# rien à voir avec l'accès au dépôt.
import os
from getpass import getpass

os.makedirs("data", exist_ok=True)

files_needed = [
    "sim_category_daily.csv",       # partie 1 : agrégat quotidien par catégorie (~10.5k lignes)
    "sim_facture_detail_raw.csv",   # partie 2 : lignes de facture brutes (~567k lignes)
]

token = None
for fname in files_needed:
    csv_path = f"data/{fname}"
    if os.path.exists(csv_path) and os.path.getsize(csv_path) > 0:
        print(f"Déjà présent sur cette VM : {csv_path} ({os.path.getsize(csv_path)} octets) — rien à faire")
        continue
    if token is None:
        token = getpass("GitHub token (repo scope, dépôt privé) : ")
    url = f"https://raw.githubusercontent.com/youcefsnoussi/dashboard/ai-forecast-gpu-notebook/notebooks/data/{fname}"
    exit_code = os.system(f'curl -sfL -H "Authorization: token {token}" "{url}" -o "{csv_path}"')
    if exit_code != 0 or not os.path.exists(csv_path) or os.path.getsize(csv_path) == 0:
        raise RuntimeError(
            f"Échec du téléchargement de {fname} — vérifie que le token a le scope "
            "'repo' et que la branche/le chemin ci-dessus sont corrects."
        )
    print(f"Téléchargé : {csv_path} ({os.path.getsize(csv_path)} octets)")
del token


# Prévision de la demande — recherche d'hyperparamètres sur GPU

Ce notebook reprend exactement le pipeline déployé dans l'application (même feature
engineering, même découpage temporel, même donnée réelle — `commercial.facture_detail`
de Groupe SIM, agrégée par catégorie et par jour) et va plus loin : une vraie recherche
de grille sur GPU avec XGBoost, comparée honnêtement contre le modèle actuellement en
production (Random Forest, non réglé) et contre deux références naïves.

**Sur Colab : Exécution → Modifier le type d'exécution → GPU** avant de lancer quoi que
ce soit ci-dessous.

**Honnêteté sur la taille des données avant de commencer :** ce CSV fait ~10 500 lignes
(agrégat quotidien par catégorie, pas les transactions individuelles). Un GPU n'apporte
aucun gain de vitesse mesurable sur un jeu de cette taille — XGBoost tournerait presque
aussi vite sur CPU. Ce notebook est câblé pour le GPU parce que la prochaine étape
naturelle est de refaire cette recherche sur les données ligne par ligne
(`facture_detail` brut, potentiellement des millions de lignes), là où le GPU devient
réellement nécessaire. Le résultat de la recherche de grille lui-même reste valable
indépendamment du matériel utilisé.


In [2]:
!nvidia-smi || echo "Pas de GPU détecté dans cette session Colab — Exécution > Modifier le type d'exécution > GPU"


Sun Sep  6 17:11:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   34C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "xgboost", "scikit-learn"], check=True)

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
import matplotlib.pyplot as plt

print("xgboost", xgb.__version__)

# get_params() only echoes back what you pass in - it does NOT prove the GPU was
# actually used. "tree_method=gpu_hist" is a deprecated alias on recent XGBoost and
# can silently fall back to CPU. Real proof: fit a small model with device="cuda"
# and check the booster's own device attribute after training, not before.
_probe = xgb.XGBRegressor(tree_method="hist", device="cuda", n_estimators=50)
_probe.fit(np.random.rand(2000, 8), np.random.rand(2000))
_actual_device = _probe.get_booster().save_config()
import json
_device_used = json.loads(_actual_device)["learner"]["generic_param"]["device"]
print("Device actually used by a trained booster:", _device_used)
assert _device_used.startswith("cuda"), "XGBoost did NOT train on GPU - check Runtime > Change runtime type > GPU"


xgboost 3.4.1
Device actually used by a trained booster: cuda:0


## 1. Charger la donnée réelle

Même source que le modèle déployé : quantité facturée, sommée par catégorie et par
jour, extraite de `proforma_cmd_bl_fact.facture_detail` ⋈ `facture` ⋈ `article` ⋈
`produit` ⋈ `sous_category_produit` ⋈ `category_produit` sur la copie locale restaurée
de la base `commercial`. Le fichier `data/sim_category_daily.csv` de ce dépôt est cet
export brut, sans aucun retraitement.


In [4]:
df = pd.read_csv("data/sim_category_daily.csv")
df["jour"] = pd.to_datetime(df["jour"], format="%d/%m/%Y")
df = df.groupby(["category", "jour"], as_index=False)["qty"].sum()

print(df["jour"].min().date(), "->", df["jour"].max().date(), f"({len(df)} lignes catégorie-jour)")
df.head()


2021-01-01 -> 2026-09-06 (10547 lignes catégorie-jour)


,category,jour,qty
0,Big bag,2025-05-31,416.0
1,Big bag,2025-06-30,290.0
2,Big bag,2025-09-30,862.0
3,Big bag,2025-10-30,265.0
4,Big bag,2025-11-30,166.0


## 2. Feature engineering — identique au pipeline déployé

Jours de la semaine, mois, week-end, puis les décalages (`lag_7/14/28`) et moyennes
mobiles (`roll_mean_7/28`) qui donnent au modèle une notion de "ce qui s'est passé
récemment". Tout est décalé (`shift`) pour qu'aucune ligne ne voie sa propre valeur
future — c'est la même discipline que le prototype de prévision original sur données
Kaggle.


In [5]:
counts = df.groupby("category")["jour"].count()
categories = counts[counts >= 120].index.tolist()
print("Catégories retenues (>=120 jours d'historique) :", categories)

full_range = pd.date_range(df["jour"].min(), df["jour"].max(), freq="D")

frames = []
for cat in categories:
    s = df[df["category"] == cat].set_index("jour")["qty"].reindex(full_range, fill_value=0)
    f = pd.DataFrame({"jour": full_range, "category": cat, "qty": s.values})
    f["dow"] = f["jour"].dt.dayofweek
    f["month"] = f["jour"].dt.month
    f["weekend"] = (f["dow"] >= 5).astype(int)
    f["lag_7"] = f["qty"].shift(7)
    f["lag_14"] = f["qty"].shift(14)
    f["lag_28"] = f["qty"].shift(28)
    f["roll_mean_7"] = f["qty"].shift(1).rolling(7).mean()
    f["roll_mean_28"] = f["qty"].shift(1).rolling(28).mean()
    frames.append(f)

data = pd.concat(frames, ignore_index=True).dropna()
data_encoded = pd.get_dummies(data, columns=["category"], prefix="cat")
feature_cols = [c for c in data_encoded.columns if c not in ("jour", "qty")]
print(f"{len(data_encoded)} lignes après feature engineering, {len(feature_cols)} colonnes de features")


Catégories retenues (>=120 jours d'historique) : ['Couscous', 'Farine', 'Pain', 'Pates', 'Semoule', 'Son', 'Transport']
14329 lignes après feature engineering, 15 colonnes de features


## 3. Découpage temporel — identique au pipeline déployé

Coupure = dernière date observée moins 56 jours (8 semaines). Entraînement sur tout ce
qui précède, test sur tout ce qui suit — jamais l'inverse, jamais mélangé.


In [6]:
cutoff = data_encoded["jour"].max() - pd.Timedelta(days=56)
train = data_encoded[data_encoded["jour"] <= cutoff].reset_index(drop=True)
test  = data_encoded[data_encoded["jour"] >  cutoff].reset_index(drop=True)

print("Coupure :", cutoff.date())
print(f"Train : {len(train)} lignes  |  Test : {len(test)} lignes (jamais vues à l'entraînement)")

cat_cols = [c for c in data_encoded.columns if c.startswith("cat_")]
test_category = test[cat_cols].idxmax(axis=1).str.replace("cat_", "", regex=False)

def mape(actual, pred):
    actual, pred = np.asarray(actual, float), np.asarray(pred, float)
    mask = actual != 0
    return float((np.abs(pred[mask] - actual[mask]) / actual[mask]).mean() * 100) if mask.any() else float("nan")

def mae(actual, pred):
    actual, pred = np.asarray(actual, float), np.asarray(pred, float)
    return float(np.abs(pred - actual).mean())

def score_by_category(pred_array, label):
    rows = []
    for cat in categories:
        m = test_category == cat
        if m.sum() == 0 or test.loc[m, "qty"].sum() == 0:
            continue
        rows.append({"category": cat, "model": label,
                     "mae": round(mae(test.loc[m, "qty"], pred_array[m.values]), 1),
                     "mape": round(mape(test.loc[m, "qty"], pred_array[m.values]), 1)})
    return pd.DataFrame(rows)


Coupure : 2026-07-12
Train : 13937 lignes  |  Test : 392 lignes (jamais vues à l'entraînement)


## 4. Références naïves — le plancher à battre

Pas des modèles, des suppositions simples : "comme il y a 7 jours" et "la moyenne des
28 derniers jours". Un modèle qui ne bat pas ça n'a pas gagné le droit d'être plus
compliqué.


In [7]:
naive_lag7  = test["lag_7"].values
naive_avg28 = test["roll_mean_28"].values

results = pd.concat([
    score_by_category(naive_lag7,  "Naïf (lag-7)"),
    score_by_category(naive_avg28, "Naïf (moyenne 28j)"),
])
results


,category,model,mae,mape
0,Couscous,Naïf (lag-7),215.1,24.4
1,Farine,Naïf (lag-7),133.4,11.2
2,Pain,Naïf (lag-7),5445.2,100.0
3,Pates,Naïf (lag-7),405.3,21.8
4,Semoule,Naïf (lag-7),966.3,39.9
5,Son,Naïf (lag-7),1375.0,51.9
0,Couscous,Naïf (moyenne 28j),318.7,19.0
1,Farine,Naïf (moyenne 28j),352.8,12.2
2,Pain,Naïf (moyenne 28j),5617.1,537.4
3,Pates,Naïf (moyenne 28j),690.8,19.7


## 5. Le modèle actuellement en production — Random Forest, non réglé

Mêmes hyperparamètres que `train_sim_forecast.py`, le script qui alimente la page
*Prévision IA* de l'application aujourd'hui. Sert de référence : est-ce que la
recherche de grille apporte un vrai gain, ou est-ce qu'on retunerait juste le même
résultat avec plus d'étapes ?


In [8]:
rf_deployed = RandomForestRegressor(n_estimators=300, max_depth=10, min_samples_leaf=3,
                                     random_state=42, n_jobs=-1)
rf_deployed.fit(train[feature_cols], train["qty"])
pred_rf = np.clip(rf_deployed.predict(test[feature_cols]), 0, None)

results = pd.concat([results, score_by_category(pred_rf, "Random Forest (déployé)")])
results


,category,model,mae,mape
0,Couscous,Naïf (lag-7),215.1,24.4
1,Farine,Naïf (lag-7),133.4,11.2
2,Pain,Naïf (lag-7),5445.2,100.0
3,Pates,Naïf (lag-7),405.3,21.8
4,Semoule,Naïf (lag-7),966.3,39.9
5,Son,Naïf (lag-7),1375.0,51.9
0,Couscous,Naïf (moyenne 28j),318.7,19.0
1,Farine,Naïf (moyenne 28j),352.8,12.2
2,Pain,Naïf (moyenne 28j),5617.1,537.4
3,Pates,Naïf (moyenne 28j),690.8,19.7


## 6. Recherche de grille sur GPU — XGBoost

`device="cuda"` (l'API actuelle — `tree_method="gpu_hist"` seul est un alias déprécié
qui peut retomber sur CPU sans prévenir sur les versions récentes) pousse chaque arbre
sur le GPU. La validation croisée utilise `TimeSeriesSplit`, pas un k-fold classique —
un k-fold mélangerait des dates futures dans les plis d'entraînement, exactement la
fuite qu'on a évité en étape 3. Grille : 3 × 4 × 3 × 2 × 2 = **144 combinaisons**,
chacune évaluée sur 3 découpes temporelles successives, soit 432 entraînements XGBoost.
`n_jobs=1` côté `GridSearchCV` est volontaire : c'est le GPU qui doit paralléliser
l'entraînement de chaque arbre, pas le CPU qui lance plusieurs XGBoost en même temps
et se dispute l'unique GPU disponible sur Colab.


In [9]:
param_grid = {
    "n_estimators":     [200, 400, 800],
    "max_depth":        [4, 6, 8, 10],
    "learning_rate":    [0.01, 0.05, 0.1],
    "subsample":        [0.7, 1.0],
    "colsample_bytree": [0.7, 1.0],
}

xgb_gpu = xgb.XGBRegressor(tree_method="hist", device="cuda",
                            objective="reg:squarederror", random_state=42, verbosity=0)

tscv = TimeSeriesSplit(n_splits=3)

search = GridSearchCV(xgb_gpu, param_grid, cv=tscv,
                       scoring="neg_mean_absolute_error", n_jobs=1, verbose=1)

search.fit(train[feature_cols], train["qty"])

print("Meilleurs hyperparamètres :", search.best_params_)
print("Meilleur MAE en validation croisée :", round(-search.best_score_, 1))

# Same proof as the probe above, on the actual best model this time - not optional.
_device_used = json.loads(search.best_estimator_.get_booster().save_config())["learner"]["generic_param"]["device"]
print("Device used by the winning model:", _device_used)
assert _device_used.startswith("cuda"), "The grid search itself ran on CPU, not GPU"


Fitting 3 folds for each of 144 candidates, totalling 432 fits


KeyboardInterrupt: 

In [ ]:
best_xgb = search.best_estimator_
pred_xgb = np.clip(best_xgb.predict(test[feature_cols]), 0, None)

results = pd.concat([results, score_by_category(pred_xgb, "XGBoost (GPU, réglé)")])
results.pivot(index="category", columns="model", values="mape").round(1)


## 7. Le vrai test : est-ce que XGBoost bat le modèle déjà en production ?

Pas "est-ce que le MAPE a l'air bien" — est-ce qu'il bat les deux références naïves
**et** le Random Forest déployé, catégorie par catégorie. Si non sur certaines
catégories, ce tableau le dit aussi, honnêtement.


In [ ]:
pivot_mae = results.pivot(index="category", columns="model", values="mae").round(1)
pivot_mae = pivot_mae[["Naïf (lag-7)", "Naïf (moyenne 28j)", "Random Forest (déployé)", "XGBoost (GPU, réglé)"]]
pivot_mae


In [ ]:
fig, axes = plt.subplots(len(categories), 1, figsize=(11, 2.6 * len(categories)), sharex=False)
for ax, cat in zip(axes, categories):
    m = (test_category == cat).values
    if m.sum() == 0:
        continue
    dates = test.loc[m, "jour"]
    ax.plot(dates, test.loc[m, "qty"], color="#8792a6", lw=1.6, label="Réel")
    ax.plot(dates, pred_xgb[m], color="#2a78d6", lw=1.6, label="Prévu (XGBoost GPU)")
    ax.set_title(cat, loc="left", fontsize=11, fontweight="bold")
    ax.legend(fontsize=8, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("xgboost_actual_vs_predicted.png", dpi=140)
plt.show()


## 8. Sauvegarder — pour rapatrier dans l'application si le gain est réel

Écrit un modèle (`best_xgb_model.json`) et un CSV de prédictions dans le même format
que `ai_forecast.demand_test` déjà utilisé par la page *Prévision IA* — remplaçable
directement si les résultats ci-dessus justifient de changer de modèle en production.


In [ ]:
best_xgb.save_model("best_xgb_model.json")

out = test[["jour"]].copy()
out["category"] = test_category.values
out["actual"] = test["qty"].values
out["predicted"] = pred_xgb
out["date"] = out["jour"].dt.strftime("%Y-%m-%d")
out[["category", "date", "actual", "predicted"]].to_csv("xgboost_predictions.csv", index=False)

print("Écrit : best_xgb_model.json, xgboost_predictions.csv, xgboost_actual_vs_predicted.png")


## Conclusion — à remplir après exécution

Ce notebook calcule tout à l'exécution ; les chiffres exacts dépendent du run. Avant de
proposer de remplacer le Random Forest en production, vérifier explicitement :

- XGBoost bat-il le Random Forest déployé sur **la majorité** des catégories, ou
  seulement une ou deux (auquel cas le gain est peut-être du bruit) ?
- Le meilleur `max_depth`/`n_estimators` trouvé par la grille est-il proche des bords de
  la grille définie à l'étape 6 ? Si oui, la grille était mal centrée — il faut l'élargir
  et relancer, pas conclure.
- Est-ce que le gain (s'il existe) justifie la complexité opérationnelle d'un second
  modèle à maintenir, ou est-ce que le Random Forest actuel — plus simple, déjà en
  production, déjà compris — reste le meilleur choix malgré un MAPE légèrement plus
  élevé ? Un modèle plus précis de 2 points de MAPE mais plus fragile n'est pas
  automatiquement une amélioration.


# Partie 2 — données ligne par ligne (566 970 lignes), pour vraiment solliciter le GPU

La partie 1 ci-dessus est honnête sur sa limite : ~10 500 lignes agrégées ne
justifient pas un GPU. Cette partie reprend la même base `commercial`, mais sans
agréger — `proforma_cmd_bl_fact.facture_detail` ⋈ `facture` ⋈ `client` ⋈ `article` ⋈
`produit` ⋈ `sous_category_produit` ⋈ `category_produit`, une ligne par article vendu
sur une facture réelle : **566 970 lignes**, 434 clients réels, 9 catégories, sur
`data/sim_facture_detail_raw.csv`.

**Nouvelle tâche, plus riche** : prédire la quantité de la **prochaine commande** d'un
client donné dans une catégorie donnée, à partir de son propre historique d'achats
(décalages temporels par client × catégorie) — pas juste "la demande totale d'hier".
C'est un problème avec beaucoup plus de structure (identité du client, sa région, son
historique propre) qu'une simple série temporelle agrégée.

**Honnêteté à nouveau, dans l'autre sens cette fois** : même à 567k lignes et ~15
colonnes, on reste loin de ce qui sature réellement 22 Go de VRAM (ça demanderait des
millions de lignes avec des embeddings à très large cardinalité, ou un modèle
séquence/image/texte). Ce qui suit *augmente réellement* la charge GPU par rapport à
la partie 1 — plus de données, des arbres plus profonds, un vrai réseau de neurones
avec des embeddings et un grand batch — sans jamais gonfler artificiellement le
calcul pour le plaisir d'occuper de la mémoire. Les chiffres de mémoire/utilisation
GPU affichés plus bas sont mesurés, pas mis en scène.


In [ ]:
raw = pd.read_csv("data/sim_facture_detail_raw.csv")
raw["facture_date"] = pd.to_datetime(raw["facture_date"], format="%d/%m/%Y")
raw = raw.dropna(subset=["client_id"]).copy()
raw["client_id"] = raw["client_id"].astype(int)
raw = raw.sort_values(["client_id", "category", "facture_date"]).reset_index(drop=True)

print(f"{len(raw)} lignes brutes, {raw['client_id'].nunique()} clients, "
      f"{raw['category'].nunique()} catégories, {raw['facture_date'].min().date()} -> "
      f"{raw['facture_date'].max().date()}")
raw.head()


## 9. Feature engineering ligne par ligne

Pour chaque paire (client, catégorie), on trie chronologiquement ses commandes et on
calcule ses trois dernières quantités (`lag_1/2/3`), sa moyenne mobile sur ses 3
dernières commandes, et le nombre de jours écoulés depuis sa commande précédente dans
cette catégorie. Tout est décalé (`shift`) — un client sans historique suffisant
(moins de 3 commandes passées) est exclu, exactement comme les catégories avec
`<120` jours l'étaient en partie 1. `client_avg_qty` est un encodage de cible
**calculé uniquement sur le train** puis appliqué au test — le calculer sur
l'ensemble complet aurait fait fuiter de l'information du futur dans le passé.


In [ ]:
g = raw.groupby(["client_id", "category"])
raw["lag_1"] = g["quantite"].shift(1)
raw["lag_2"] = g["quantite"].shift(2)
raw["lag_3"] = g["quantite"].shift(3)
raw["roll_mean_3"] = g["quantite"].shift(1).rolling(3).mean().reset_index(level=[0, 1], drop=True)
raw["days_since_last"] = g["facture_date"].diff().dt.days
raw["dow"] = raw["facture_date"].dt.dayofweek
raw["month"] = raw["facture_date"].dt.month

row = raw.dropna(subset=["lag_1", "lag_2", "lag_3", "roll_mean_3", "days_since_last"]).reset_index(drop=True)
row["wilaya_id"] = row["wilaya_id"].fillna(-1).astype(int)
row["client_category_id"] = row["client_category_id"].fillna(-1).astype(int)

print(f"{len(row)} lignes après feature engineering "
      f"({raw['client_id'].nunique() - row['client_id'].nunique()} clients exclus, historique insuffisant)")
row[["client_id", "category", "facture_date", "quantite", "lag_1", "lag_2", "roll_mean_3", "days_since_last"]].head()


## 10. Découpage temporel et encodages — même discipline qu'en partie 1

Coupure à 56 jours avant la dernière commande observée, comme en partie 1. Catégorie,
wilaya et catégorie-client sont encodées en one-hot pour XGBoost (peu de modalités,
pas de fuite possible) ; `client_id` est trop cardinal (434 valeurs) pour du one-hot,
donc XGBoost reçoit `client_avg_qty` (moyenne du client, calculée sur le train
seulement) à la place, tandis que le réseau de neurones plus bas apprendra un
embedding dédié pour `client_id` directement.


In [ ]:
cutoff_row = row["facture_date"].max() - pd.Timedelta(days=56)
train_row = row[row["facture_date"] <= cutoff_row].reset_index(drop=True)
test_row  = row[row["facture_date"] >  cutoff_row].reset_index(drop=True)

print("Coupure :", cutoff_row.date())
print(f"Train : {len(train_row)} lignes  |  Test : {len(test_row)} lignes (jamais vues à l'entraînement)")

# Target encoding du client - calculé UNIQUEMENT sur le train.
global_mean_qty = train_row["quantite"].mean()
client_avg = train_row.groupby("client_id")["quantite"].mean()
train_row = train_row.assign(client_avg_qty=train_row["client_id"].map(client_avg))
test_row  = test_row.assign(client_avg_qty=test_row["client_id"].map(client_avg).fillna(global_mean_qty))

numeric_cols = ["lag_1", "lag_2", "lag_3", "roll_mean_3", "days_since_last",
                "dow", "month", "prix_u_ht", "client_avg_qty"]

both = pd.concat([train_row.assign(_split="train"), test_row.assign(_split="test")], ignore_index=True)
both_encoded = pd.get_dummies(both, columns=["category", "wilaya_id", "client_category_id"],
                               prefix=["cat", "wil", "ccat"])
onehot_cols = [c for c in both_encoded.columns if c.startswith(("cat_", "wil_", "ccat_"))]
feature_cols_row = numeric_cols + onehot_cols

train_enc = both_encoded[both_encoded["_split"] == "train"].reset_index(drop=True)
test_enc  = both_encoded[both_encoded["_split"] == "test"].reset_index(drop=True)
print(f"{len(feature_cols_row)} colonnes de features pour XGBoost (dont {len(onehot_cols)} one-hot)")


## 11. Référence naïve ligne par ligne

"La prochaine commande ressemblera à la précédente" (`lag_1`). Le plancher à battre
pour les deux modèles GPU ci-dessous.


In [ ]:
def mae_row(actual, pred):
    return float(np.abs(np.asarray(pred, float) - np.asarray(actual, float)).mean())

def mape_row(actual, pred):
    actual, pred = np.asarray(actual, float), np.asarray(pred, float)
    mask = actual != 0
    return float((np.abs(pred[mask] - actual[mask]) / actual[mask]).mean() * 100) if mask.any() else float("nan")

results_row = pd.DataFrame([{
    "model": "Naïf (dernière commande)",
    "mae": round(mae_row(test_enc["quantite"], test_enc["lag_1"]), 2),
    "mape": round(mape_row(test_enc["quantite"], test_enc["lag_1"]), 1),
    "gpu_mem_used_mb": None,
}])
results_row


## 12. XGBoost sur 567k lignes — grille élargie, vraie charge GPU

Même logique qu'en partie 1 (`device="cuda"`, `TimeSeriesSplit`, `n_jobs=1` côté
`GridSearchCV`), mais la grille est élargie sur les axes qui pèsent réellement sur la
mémoire GPU : `max_depth` jusqu'à 14 (arbres plus profonds = plus de nœuds à stocker
sur le device), `max_bin` porté à 512 (les histogrammes GPU d'XGBoost scalent avec le
nombre de bins × le nombre de lignes — c'est le levier direct pour occuper plus de
VRAM, pas un raccourci). Grille : 3 × 5 × 3 × 2 = **90 combinaisons** × 3 découpes
temporelles = 270 entraînements, chacun sur 442k lignes de train (31x plus qu'en
partie 1). On imprime la mémoire GPU réellement utilisée juste après, mesurée via
`nvidia-smi`, pas devinée.


In [ ]:
def gpu_mem_used_mb():
    out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                          capture_output=True, text=True, check=True)
    return int(out.stdout.strip().splitlines()[0])

param_grid_row = {
    "n_estimators":  [500, 1000, 2000],
    "max_depth":     [6, 8, 10, 12, 14],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample":     [0.8, 1.0],
}

xgb_gpu_row = xgb.XGBRegressor(tree_method="hist", device="cuda", max_bin=512,
                                objective="reg:squarederror", random_state=42, verbosity=0)

tscv_row = TimeSeriesSplit(n_splits=3)

mem_before = gpu_mem_used_mb()
search_row = GridSearchCV(xgb_gpu_row, param_grid_row, cv=tscv_row,
                           scoring="neg_mean_absolute_error", n_jobs=1, verbose=1)
search_row.fit(train_enc[feature_cols_row], train_enc["quantite"])
mem_during_search = gpu_mem_used_mb()

print("Meilleurs hyperparamètres :", search_row.best_params_)
print("Meilleur MAE en validation croisée :", round(-search_row.best_score_, 2))
print(f"Mémoire GPU utilisée : {mem_before} MiB avant -> {mem_during_search} MiB pendant "
      f"(pic mesuré juste après l'entraînement du meilleur modèle)")

_device_used = json.loads(search_row.best_estimator_.get_booster().save_config())["learner"]["generic_param"]["device"]
print("Device utilisé par le modèle gagnant :", _device_used)
assert _device_used.startswith("cuda"), "La recherche a tourné sur CPU, pas sur GPU"


In [ ]:
best_xgb_row = search_row.best_estimator_
pred_xgb_row = np.clip(best_xgb_row.predict(test_enc[feature_cols_row]), 0, None)

results_row = pd.concat([results_row, pd.DataFrame([{
    "model": "XGBoost (GPU, grille élargie, 567k lignes)",
    "mae": round(mae_row(test_enc["quantite"], pred_xgb_row), 2),
    "mape": round(mape_row(test_enc["quantite"], pred_xgb_row), 1),
    "gpu_mem_used_mb": mem_during_search,
}])], ignore_index=True)
results_row


## 13. Réseau de neurones (PyTorch, embeddings + MLP large) — le vrai test de VRAM

XGBoost sur données tabulaires n'a jamais vraiment besoin de beaucoup de VRAM, même
avec une grille élargie — c'est un modèle par nature léger en mémoire. Un réseau de
neurones avec des tables d'embeddings (client, catégorie, wilaya, catégorie-client)
et des couches larges (1024 → 512 → 256), entraîné en grand batch (16 384) et en
précision mixte (`torch.autocast`, qui utilise les tensor cores de l'L4), est le
choix honnête si l'objectif est de solliciter davantage le GPU. **Attendu, et dit
clairement à l'avance** : sur 567k lignes et une tâche à faible dimensionnalité, ce
réseau n'a aucune raison de battre XGBoost en précision — l'intérêt ici est la charge
GPU réellement mesurée, pas une meilleure prévision. S'il perd sur le MAE/MAPE, ce
sera écrit tel quel dans la conclusion, pas caché.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

assert torch.cuda.is_available(), "Pas de GPU CUDA visible pour PyTorch - Exécution > Modifier le type d'exécution > GPU"
device = torch.device("cuda")

# Encodage entier pour les embeddings - vocabulaire figé sur le train, "inconnu" = index 0
cat_specs = {
    "client_id":          train_row["client_id"],
    "category":           train_row["category"],
    "wilaya_id":          train_row["wilaya_id"],
    "client_category_id": train_row["client_category_id"],
}
vocabs = {name: {v: i + 1 for i, v in enumerate(sorted(s.unique()))} for name, s in cat_specs.items()}

def encode_cats(df):
    return np.stack([df[name].map(vocabs[name]).fillna(0).astype(int).values for name in cat_specs], axis=1)

num_scaler_mean = train_row[["lag_1", "lag_2", "lag_3", "roll_mean_3", "days_since_last", "prix_u_ht"]].mean()
num_scaler_std  = train_row[["lag_1", "lag_2", "lag_3", "roll_mean_3", "days_since_last", "prix_u_ht"]].std().replace(0, 1)
num_cols_nn = ["lag_1", "lag_2", "lag_3", "roll_mean_3", "days_since_last", "prix_u_ht"]

def encode_nums(df):
    return ((df[num_cols_nn] - num_scaler_mean) / num_scaler_std).values.astype(np.float32)

X_train_cat = torch.tensor(encode_cats(train_row), dtype=torch.long)
X_train_num = torch.tensor(encode_nums(train_row), dtype=torch.float32)
y_train_nn  = torch.tensor(train_row["quantite"].values, dtype=torch.float32).unsqueeze(1)

X_test_cat = torch.tensor(encode_cats(test_row), dtype=torch.long)
X_test_num = torch.tensor(encode_nums(test_row), dtype=torch.float32)
y_test_nn  = torch.tensor(test_row["quantite"].values, dtype=torch.float32).unsqueeze(1)

vocab_sizes = [len(v) + 1 for v in vocabs.values()]
print("Tailles de vocabulaire (client, catégorie, wilaya, catégorie-client) :", vocab_sizes)
print(f"Train : {len(X_train_cat)} lignes  |  Test : {len(X_test_cat)} lignes")


In [ ]:
class DemandNet(nn.Module):
    def __init__(self, vocab_sizes, n_numeric, emb_dim=32, hidden=(1024, 512, 256)):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(v, emb_dim) for v in vocab_sizes])
        in_dim = emb_dim * len(vocab_sizes) + n_numeric
        layers = []
        for h in hidden:
            layers += [nn.Linear(in_dim, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(0.2)]
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.mlp = nn.Sequential(*layers)

    def forward(self, x_cat, x_num):
        embedded = torch.cat([emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)], dim=1)
        return self.mlp(torch.cat([embedded, x_num], dim=1))

model_nn = DemandNet(vocab_sizes, n_numeric=X_train_num.shape[1]).to(device)
n_params = sum(p.numel() for p in model_nn.parameters())
print(f"{n_params:,} paramètres")

train_loader = DataLoader(TensorDataset(X_train_cat, X_train_num, y_train_nn),
                           batch_size=16384, shuffle=True, drop_last=False)

optimizer = torch.optim.Adam(model_nn.parameters(), lr=1e-3)
loss_fn = nn.L1Loss()  # MAE directement comme fonction de perte
scaler = torch.amp.GradScaler("cuda")

torch.cuda.reset_peak_memory_stats()
epochs = 25
for epoch in range(epochs):
    model_nn.train()
    total_loss = 0.0
    for xb_cat, xb_num, yb in train_loader:
        xb_cat, xb_num, yb = xb_cat.to(device), xb_num.to(device), yb.to(device)
        optimizer.zero_grad()
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            pred = model_nn(xb_cat, xb_num)
            loss = loss_fn(pred, yb)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * len(yb)
    if epoch % 5 == 0 or epoch == epochs - 1:
        print(f"Époque {epoch+1}/{epochs} - MAE train : {total_loss / len(X_train_cat):.2f}")

peak_mem_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
print(f"\nPic de mémoire GPU allouée par PyTorch pendant l'entraînement : {peak_mem_mb:.0f} MiB")
print(f"(à comparer aux {mem_during_search} MiB mesurés pendant la recherche XGBoost ci-dessus)")


In [ ]:
model_nn.eval()
with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16):
    pred_nn = model_nn(X_test_cat.to(device), X_test_num.to(device)).float().cpu().numpy().ravel()
pred_nn = np.clip(pred_nn, 0, None)

results_row = pd.concat([results_row, pd.DataFrame([{
    "model": "Réseau de neurones (GPU, embeddings, batch 16384)",
    "mae": round(mae_row(test_enc["quantite"], pred_nn), 2),
    "mape": round(mape_row(test_enc["quantite"], pred_nn), 1),
    "gpu_mem_used_mb": round(peak_mem_mb),
}])], ignore_index=True)
results_row


## Conclusion — Partie 2, à remplir après exécution

Comme en partie 1, les chiffres exacts dépendent du run. Avant de tirer une
conclusion :

- **Sur la précision** : le réseau de neurones bat-il réellement le XGBoost row-level
  et la référence naïve, ou seulement l'un des deux ? Sur ~442k lignes d'entraînement
  et une tâche à peu de features, il est **normal et attendu** que XGBoost reste
  compétitif voire meilleur — un réseau de neurones n'est pas automatiquement
  supérieur, surtout sur données tabulaires de cette taille. Si `results_row` le
  montre, c'est le résultat honnête, pas un échec du notebook.
- **Sur la VRAM** : comparer les trois valeurs de `gpu_mem_used_mb` ci-dessus.
  L'écart entre la partie 1 (~290 MiB observés en usage réel) et cette partie doit
  être net et mesuré, pas supposé. S'il reste faible malgré tout, la conclusion
  honnête est la même qu'en partie 1 : la vraie limite n'est pas le code mais la
  taille de la donnée — SIM produit des centaines de milliers de lignes de ventes,
  pas des millions, et un L4 de 22 Go est dimensionné pour des charges bien plus
  lourdes (LLMs, vision, séquences longues) que la prévision de demande tabulaire.
- **Sur la suite** : si un des deux modèles GPU bat le Random Forest déployé de façon
  nette et reproductible, l'étape suivante est de le rapatrier dans
  `ai_forecast.demand_test` / `demand_metrics` comme indiqué en partie 1 — jamais sur
  la seule base d'un MAPE qui a l'air bien à l'œil.
